In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

auto_mpg = fetch_ucirepo(id=9)
data = pd.concat([auto_mpg.data.features, auto_mpg.data.targets], axis=1).dropna()

X = data[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']]
y = data['mpg']

scaler = MinMaxScaler()
X[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']] = scaler.fit_transform(
    X[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']]
)

years = torch.tensor(X['model_year'].values, dtype=torch.float32)
boundaries = torch.tensor([73, 76, 79])
X['year_bucket'] = torch.bucketize(years, boundaries, right=False)
X = X.drop('model_year', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = pd.get_dummies(X_train, columns=['origin'], prefix='origin')
X_test = pd.get_dummies(X_test, columns=['origin'], prefix='origin')

X_train = X_train.astype(float)
X_test = X_test.astype(float)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)
epochs = 200

for epoch in range(epochs):
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = model(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")

Эпоха [20/200] — Потери (Loss): 573.0052
Эпоха [40/200] — Потери (Loss): 297.4395
Эпоха [60/200] — Потери (Loss): 49.6148
Эпоха [80/200] — Потери (Loss): 38.1998
Эпоха [100/200] — Потери (Loss): 29.9929
Эпоха [120/200] — Потери (Loss): 24.2551
Эпоха [140/200] — Потери (Loss): 20.5570
Эпоха [160/200] — Потери (Loss): 18.2867
Эпоха [180/200] — Потери (Loss): 16.8485
Эпоха [200/200] — Потери (Loss): 15.8677

Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 15.8771
MAE (средняя абсолютная ошибка): 3.0638


C:\Users\CifronPro\AppData\Local\Temp\ipykernel_13928\930848087.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']] = scaler.fit_transform(
